## 🎯 Learning Objectives
* Understand the limitations of traditional word representations like one-hot encoding.
* Explain the core concepts behind static word embeddings such as word2vec (Skip-gram, CBOW) and GloVe.
* Demonstrate how to load and utilize pre-trained static word embeddings to calculate word similarity and perform analogies.
* Grasp the fundamental idea of contextual word embeddings and their advantage over static embeddings.
* Utilize a pre-trained transformer model (e.g., BERT) to extract contextual embeddings for words in different sentences.
* Compare and contrast static and contextual embeddings, identifying their respective strengths, weaknesses, and appropriate use cases.


# Embeddings: word2vec, GloVe, and Contextual Embeddings

Welcome to the fascinating world of word embeddings! In Natural Language Processing (NLP), computers don't understand words as humans do. They need numerical representations. Historically, simple methods like one-hot encoding were used, where each word was a unique vector with a '1' at its corresponding index and '0's elsewhere. While straightforward, this approach suffered from two major drawbacks:

1.  **Sparsity and High Dimensionality**: For a vocabulary of 100,000 words, each word would be a 100,000-dimensional vector, mostly zeros. This is computationally expensive and inefficient.
2.  **Lack of Semantic Relationship**: One-hot vectors are orthogonal, meaning they convey no information about the similarity or relationship between words. 


## The Dawn of Distributed Representations: Static Embeddings

To overcome these limitations, the concept of **distributed representations**, or **word embeddings**, emerged. These are dense, low-dimensional vectors where words with similar meanings are mapped to similar points in the vector space. Think of it like a sophisticated map where cities that are functionally or geographically close are also close on the map.

### word2vec (2013)

Developed by Google, word2vec revolutionized how we represent words. It's not a single algorithm but a family of models that learn word associations from a large corpus of text. The core idea is: **


You shall know a word by the company it keeps.


**

There are two main architectures:

1.  **Skip-gram**: Predicts the surrounding context words given a target word. (e.g., given 


cat


, predict 


the


, 


sat


, 


on


).
2.  **CBOW (Continuous Bag-of-Words)**: Predicts the target word given its surrounding context words. (e.g., given 


the


, 


sat


, 


on


, predict 


cat


).

Both models learn word embeddings as a side effect of this prediction task. The magic happens when these learned vectors exhibit fascinating properties, like vector arithmetic for analogies (e.g., `king - man + woman ≈ queen`).

### GloVe (Global Vectors for Word Representation, 2014)

GloVe, developed at Stanford, takes a different approach. While word2vec is a 


predictive


 model, GloVe is a 


count-based


 model. It explicitly incorporates global word-word co-occurrence statistics from a corpus. It essentially tries to learn vectors such that their dot product is proportional to the logarithm of their co-occurrence probability. This allows it to capture both local context (like word2vec) and global semantic relationships.

Both word2vec and GloVe produce **static embeddings**. This means that for any given word, its embedding vector remains the same regardless of the context in which it appears. The word 


bank


 would have the same vector whether it refers to a financial institution or a river bank.

## The Next Frontier: Contextual Embeddings

While static embeddings were a huge leap forward, their inability to handle polysemy (words with multiple meanings) and homonymy (words that sound or are spelled the same but have different meanings) was a limitation. This led to the development of **contextual embeddings**.

Contextual embeddings generate a word's representation dynamically based on the entire sentence or document it appears in. This means the word 


bank


 will have a different vector in 


I went to the **bank** to deposit money


 than in 


The boat sailed along the river **bank**.




Models like ELMo (Embeddings from Language Models), BERT (Bidirectional Encoder Representations from Transformers), GPT (Generative Pre-trained Transformer), and their successors (RoBERTa, T5, Llama, etc.) are prime examples of architectures that produce contextual embeddings. They leverage deep neural networks, particularly the Transformer architecture, to understand the intricate relationships between words in a sequence.

These models are typically pre-trained on massive text datasets (like the entire internet!) to learn general language understanding. Then, these pre-trained models can be fine-tuned for specific downstream NLP tasks, often achieving state-of-the-art performance. The embeddings extracted from intermediate layers of these models are incredibly rich in semantic and syntactic information.


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install gensim==4.3.2 transformers==4.40.0 torch==2.2.2 numpy==1.26.4 scipy==1.13.0

import gensim.downloader as api
from scipy.spatial.distance import cosine
import numpy as np

import torch
from transformers import AutoTokenizer, AutoModel

print(f"Gensim version: {gensim.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")

# --- Part 1: Static Embeddings (GloVe) ---
print("\n--- Demonstrating Static Embeddings (GloVe) ---")

# Load pre-trained GloVe embeddings
# This might take a few minutes the first time as it downloads the model.
# We'll use a smaller version for demonstration: 'glove-wiki-gigaword-50'
# Other options include 'glove-wiki-gigaword-100', 'glove-twitter-25', etc.
try:
    glove_vectors = api.load("glove-wiki-gigaword-50")
    print("GloVe embeddings loaded successfully.")
except Exception as e:
    print(f"Error loading GloVe embeddings: {e}")
    print("Please check your internet connection or try a different model.")
    glove_vectors = None # Set to None to prevent further errors if loading fails

if glove_vectors:
    # Get vector for a word
    word1 = "king"
    word2 = "queen"
    word3 = "man"
    word4 = "woman"
    word5 = "apple"

    print(f"\nVector for '{word1}': {glove_vectors[word1][:5]}... (first 5 dimensions)")

    # Calculate similarity between words (cosine similarity)
    similarity_king_queen = 1 - cosine(glove_vectors[word1], glove_vectors[word2])
    similarity_king_apple = 1 - cosine(glove_vectors[word1], glove_vectors[word5])

    print(f"Similarity between '{word1}' and '{word2}': {similarity_king_queen:.4f}")
    print(f"Similarity between '{word1}' and '{word5}': {similarity_king_apple:.4f}")

    # Perform word analogies (e.g., king - man + woman = queen)
    try:
        result = glove_vectors.most_similar(positive=[word1, word4], negative=[word3], topn=1)
        print(f"\nAnalogy: '{word1}' - '{word3}' + '{word4}' = '{result[0][0]}' (score: {result[0][1]:.4f})")
    except KeyError as e:
        print(f"Could not perform analogy: {e}. One of the words might not be in the vocabulary.")


# --- Part 2: Contextual Embeddings (BERT) ---
print("\n--- Demonstrating Contextual Embeddings (BERT) ---")

# Load pre-trained BERT tokenizer and model
# We'll use a smaller, faster BERT variant for demonstration: 'bert-base-uncased'
# This will download the model weights and configuration the first time.
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Example sentences with a polysemous word ('bank')
sentence1 = "I went to the **bank** to deposit money."
sentence2 = "The boat sailed along the river **bank**."

# Function to get contextual embedding for a specific word in a sentence
def get_contextual_embedding(text, target_word, tokenizer, model):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=True)
    input_ids = inputs["input_ids"]

    # Get model outputs (hidden states)
    with torch.no_grad():
        outputs = model(**inputs)
    # The last hidden state contains the contextual embeddings
    last_hidden_states = outputs.last_hidden_state

    # Find the token ID(s) for the target word
    # BERT tokenizes words into subwords, so we need to handle this.
    # We'll take the embedding of the first subword token for simplicity.
    target_word_tokens = tokenizer.tokenize(target_word)
    if not target_word_tokens:
        return None

    # Find the index of the first token of the target word in the input_ids
    # We need to skip special tokens like [CLS] and [SEP]
    word_indices = []
    current_idx = 1 # Start after [CLS]
    for token_id in input_ids[0][1:-1]: # Exclude [CLS] and [SEP]
        token_str = tokenizer.decode(token_id)
        if token_str == target_word_tokens[0]:
            # Check if the full target word is matched by subsequent tokens
            match = True
            for i in range(1, len(target_word_tokens)):
                if current_idx + i >= len(input_ids[0]) - 1 or \
                   tokenizer.decode(input_ids[0][current_idx + i]) != target_word_tokens[i]:
                    match = False
                    break
            if match:
                word_indices.append(current_idx)
        current_idx += 1

    if not word_indices:
        print(f"Warning: Target word '{target_word}' not found in tokenized text: {tokenizer.tokenize(text)}")
        return None

    # Get the embedding for the first occurrence of the target word's first token
    # We add 1 to the word_index because of the [CLS] token at the beginning
    target_embedding = last_hidden_states[0, word_indices[0] + 1, :].numpy()
    return target_embedding

# Get embeddings for 'bank' in both sentences
target_word = "bank"
embedding_bank_s1 = get_contextual_embedding(sentence1, target_word, tokenizer, model)
embedding_bank_s2 = get_contextual_embedding(sentence2, target_word, tokenizer, model)

if embedding_bank_s1 is not None and embedding_bank_s2 is not None:
    print(f"\nEmbedding for '{target_word}' in sentence 1 (first 5 dimensions): {embedding_bank_s1[:5]}...")
    print(f"Embedding for '{target_word}' in sentence 2 (first 5 dimensions): {embedding_bank_s2[:5]}...")

    # Calculate similarity between the two contextual embeddings of 'bank'
    similarity_contextual_bank = 1 - cosine(embedding_bank_s1, embedding_bank_s2)
    print(f"\nSimilarity between '{target_word}' in sentence 1 and sentence 2: {similarity_contextual_bank:.4f}")

    # Let's also get an embedding for a clearly different word in a similar context
    sentence3 = "I went to the **store** to buy groceries."
    embedding_store_s3 = get_contextual_embedding(sentence3, "store", tokenizer, model)

    if embedding_store_s3 is not None:
        # Compare 'bank' (financial) with 'store'
        similarity_bank_s1_store_s3 = 1 - cosine(embedding_bank_s1, embedding_store_s3)
        print(f"Similarity between 'bank' (financial) and 'store': {similarity_bank_s1_store_s3:.4f}")

        # Compare 'bank' (river) with 'store'
        similarity_bank_s2_store_s3 = 1 - cosine(embedding_bank_s2, embedding_store_s3)
        print(f"Similarity between 'bank' (river) and 'store': {similarity_bank_s2_store_s3:.4f}")

else:
    print("Could not generate contextual embeddings for comparison.")


## Interpreting the Output and Use Cases

### Static Embeddings (GloVe Output)

When you ran the code for GloVe, you observed:

*   **Vector Representation**: Each word is transformed into a dense vector (e.g., 50 dimensions for `glove-wiki-gigaword-50`). These numbers, while not directly interpretable by humans, encode the word's meaning based on its co-occurrence patterns in the training corpus.
*   **Word Similarity**: The cosine similarity score quantifies how semantically close two words are. A score closer to 1 indicates high similarity, while a score closer to 0 (or negative) indicates dissimilarity. You should have seen a higher similarity between 


king


 and 


queen


 than between 


king


 and 


apple


. This demonstrates the power of embeddings to capture semantic relationships.
*   **Word Analogies**: The analogy `king - man + woman = queen` is a classic example. By performing vector arithmetic, we can find words that maintain specific semantic relationships. This property is a hallmark of well-trained static embeddings.

**Performance Trade-offs**: Static embeddings are relatively lightweight and fast to use once loaded. They are excellent for tasks where word context isn't paramount or when computational resources are limited. Training them from scratch on a large corpus can be time-consuming, but using pre-trained models is efficient.

**Typical Use Cases**: Text classification, sentiment analysis (as features for a classifier), recommendation systems, information retrieval, simple chatbots, and as initial input layers for simpler neural networks.

### Contextual Embeddings (BERT Output)

For BERT, the key takeaway is the **dynamic nature of the embeddings**:

*   **Context-Dependent Vectors**: You should have observed that the embedding for the word 


bank


 in 


I went to the **bank** to deposit money


 is significantly different from its embedding in 


The boat sailed along the river **bank**.


 The cosine similarity between these two embeddings should be noticeably lower than if they were identical. This is the core advantage: BERT understands that 


bank


 has different meanings based on its surrounding words.
*   **Semantic Proximity**: You should also see that the 


bank


 (financial) embedding is more similar to 


store


 than to 


bank


 (river), further illustrating the contextual understanding.

**Performance Trade-offs**: Contextual embedding models like BERT are much larger and computationally more intensive than static embedding models. Extracting embeddings requires running inference through a deep neural network, which can be slower. However, the richness of the information they capture often justifies this cost.

**Typical Use Cases**: Question answering, machine translation, named entity recognition, sentiment analysis (especially for nuanced expressions), text summarization, and any task requiring deep semantic understanding of text. In 2026, these models are often the backbone of larger Generative AI applications and are frequently fine-tuned for specific enterprise tasks.

### General Considerations

*   **Pre-trained vs. Custom Training**: For most applications, using pre-trained embeddings (both static and contextual) is the standard practice. Training your own embeddings from scratch requires vast amounts of data and significant computational resources, typically only done for highly specialized domains with unique vocabulary.
*   **Dimensionality**: Embeddings typically range from 50 to 1024 dimensions, a significant reduction from one-hot encoding, making them efficient while retaining rich information.
*   **Evolution**: The field is constantly evolving. While word2vec and GloVe laid the foundation, contextual embeddings from Transformer-based models are now the state-of-the-art for most complex NLP tasks, often serving as the initial layers of larger Language Models (LLMs).


## Resources

*   **Gensim Library (for word2vec/GloVe)**:
    *   Official Documentation: [https://radimrehurek.com/gensim/](https://radimrehurek.com/gensim/)
    *   Pre-trained Models API: [https://radimrehurek.com/gensim/auto_examples/howtos/run_downloader_api.html](https://radimrehurek.com/gensim/auto_examples/howtos/run_downloader_api.html)

*   **Hugging Face Transformers Library (for BERT and other contextual models)**:
    *   Official Documentation: [https://huggingface.co/docs/transformers/index](https://huggingface.co/docs/transformers/index)
    *   BERT Model Card: [https://huggingface.co/bert-base-uncased](https://huggingface.co/bert-base-uncased)
    *   Getting Started with Tokenizers: [https://huggingface.co/docs/transformers/tokenizer_summary](https://huggingface.co/docs/transformers/tokenizer_summary)

*   **PyTorch Documentation**:
    *   `torch.nn.Embedding` (for learning embeddings from scratch): [https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

*   **Original Papers**:
    *   word2vec (Efficient Estimation of Word Representations in Vector Space): [https://arxiv.org/abs/1301.3781](https://arxiv.org/abs/1301.3781)
    *   GloVe (Global Vectors for Word Representation): [https://nlp.stanford.edu/pubs/glove.pdf](https://nlp.stanford.edu/pubs/glove.pdf)
    *   BERT (Pre-training of Deep Bidirectional Transformers for Language Understanding): [https://arxiv.org/abs/1810.04805](https://arxiv.org/abs/1810.04805)

*   **Google AI Studio / Google Cloud AI Platform**: For deploying and managing large-scale NLP models, including those based on contextual embeddings. While not directly about embeddings, it's where many production-grade applications leveraging these concepts are built.
    *   Google AI Studio: [https://ai.google.dev/](https://ai.google.dev/)
    *   Google Cloud AI Platform: [https://cloud.google.com/ai-platform](https://cloud.google.com/ai-platform)
